In [ ]:
import requests
from datetime import datetime, timedelta

API_KEY = "b2a0d07341544104b699c5576a1bf7fd"
url = "https://newsapi.org/v2/everything"

# Query parameters
query = "finance AND (stock market OR investment)"
language = "en"
page_size = 100  # max per request

# Date range setup: last 30 days
end_date = datetime.today()
start_date = end_date - timedelta(days=30)

all_articles = []

# Split the 30-day range into 3-day chunks (adjust as needed)
delta = timedelta(days=3)
current_start = start_date

while current_start < end_date:
    current_end = min(current_start + delta, end_date)
    
    page = 1
    while True:
        params = {
            "q": query,
            "language": language,
            "pageSize": page_size,
            "page": page,
            "from": current_start.strftime("%Y-%m-%d"),
            "to": current_end.strftime("%Y-%m-%d"),
            "apiKey": API_KEY
        }

        response = requests.get(url, params=params)
        data = response.json()

        if "articles" in data and data["articles"]:
            all_articles.extend(data["articles"])
            if len(data["articles"]) < page_size:
                break  # No more pages for this date range
            page += 1
        else:
            break  # No articles for this page/date range

    current_start += delta

# Extract text
news_list = []
for article in all_articles:
    text = article.get("title", "") + " " + str(article.get("description", ""))
    news_list.append(text)

print(f"Total articles fetched: {len(news_list)}")
print(news_list[:10])

In [ ]:
candidate_labels = ["Economy", "Stock Market", "Investment", "Cryptocurrency", "Politics"]

In [ ]:
from transformers import pipeline

# Load the zero-shot classification pipeline
classifier = pipeline("zero-shot-classification")

# Categorize all articles
news_categories = []

for article in news_list:
    result = classifier(article, candidate_labels)
    top_category = result['labels'][0]  # pick the label with highest score
    news_categories.append(top_category)

# Check first few results
for i in range(3):
    print(f"Article: {news_list[i]}")
    print(f"Category: {news_categories[i]}")
    print("---")

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

category_counts = Counter(news_categories)
print(category_counts)

# Bar chart
plt.bar(category_counts.keys(), category_counts.values(), color='skyblue')
plt.title("News Articles by Category")
plt.xlabel("Category")
plt.ylabel("Number of Articles")
plt.show()